In [ ]:
# ============================================
# SETUP: Imports + Load processed data
# Run order: 01_eda -> 02_feature_engineering -> 03_prepare_dataset -> 04_extension_smote_xgboost
# ============================================
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import time
import sys

from xgboost import XGBClassifier

processed_path = Path('../data/processed_split.pkl')
if not processed_path.exists():
    raise FileNotFoundError(
        'Missing ../data/processed_split.pkl. Run 03_prepare_dataset.ipynb first.'
    )

sys.path.append('../src')
from evaluation import evaluate_model, print_metrics_table

data = joblib.load(processed_path)
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']
# XGBoost la tree-based -> dung ban KHONG scale (X_train/X_test goc), khong dung X_train_scaled

print(f'Du lieu da load: X_train {X_train.shape}, X_test {X_test.shape}')
print(f'Train fraud rate: {y_train.mean():.4f}, Test fraud rate: {y_test.mean():.4f}')
print(f'Columns: {list(X_train.columns)}')

In [ ]:
# ============================================
# XGBoost — chua xu ly imbalance (default params)
# ============================================
start = time.time()
xgb_plain = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)
xgb_plain.fit(X_train, y_train)
xgb_plain_train_time = time.time() - start
print(f'Thoi gian train XGBoost (plain): {xgb_plain_train_time:.2f} giay')

xgb_plain_results = evaluate_model(
    xgb_plain, X_test, y_test, 'XGBoost (no imbalance handling)', xgb_plain_train_time
)
print(xgb_plain_results)

In [ ]:
# ============================================
# SMOTE — CHI ap dung tren train set, KHONG dung tren test set
# ============================================
from imblearn.over_sampling import SMOTE

print('Truoc SMOTE:')
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print('\nSau SMOTE (chi tren train):')
print(pd.Series(y_train_smote).value_counts())

# X_test / y_test giu nguyen, KHONG resample -> danh gia tren phan phoi that

start = time.time()
xgb_smote = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)
xgb_smote.fit(X_train_smote, y_train_smote)
xgb_smote_train_time = time.time() - start
print(f'\nThoi gian train XGBoost+SMOTE: {xgb_smote_train_time:.2f} giay')

xgb_smote_results = evaluate_model(
    xgb_smote, X_test, y_test, 'XGBoost + SMOTE', xgb_smote_train_time
)
print(xgb_smote_results)

In [ ]:
# ============================================
# So sanh: XGBoost (plain) vs XGBoost + SMOTE
# ============================================

# Bang 1: Ket qua code that (XGBoost 2 dieu kien)
xgb_comparison = print_metrics_table([xgb_plain_results, xgb_smote_results])
print('Bang so sanh XGBoost (plain) vs XGBoost + SMOTE:')
print(xgb_comparison)

# Bang 2: Doi chieu voi anchor paper (chi de tham khao / viet narrative, KHONG merge)
from evaluation import ANCHOR_PAPER_RESULTS
print('\nKet qua anchor paper (LR/KNN/DT/RF) - de doi chieu, khong merge truc tiep:')
print(pd.DataFrame(ANCHOR_PAPER_RESULTS))

# Tinh chenh lech F1 / Recall giua 2 dieu kien XGBoost - so lieu dung de viet phan Extension 1
f1_plain = xgb_plain_results['f1'] if isinstance(xgb_plain_results, dict) else xgb_plain_results.get('F1')
f1_smote = xgb_smote_results['f1'] if isinstance(xgb_smote_results, dict) else xgb_smote_results.get('F1')
recall_plain = xgb_plain_results['recall'] if isinstance(xgb_plain_results, dict) else xgb_plain_results.get('Recall')
recall_smote = xgb_smote_results['recall'] if isinstance(xgb_smote_results, dict) else xgb_smote_results.get('Recall')

print(f'\nChenh lech F1 (SMOTE - plain): {f1_smote - f1_plain:+.4f}')
print(f'Chenh lech Recall (SMOTE - plain): {recall_smote - recall_plain:+.4f}')

In [ ]:
# ============================================
# Luu models + bang ket qua cho notebook tiep theo (SHAP / cost-sensitive)
# ============================================
from pathlib import Path
import joblib

results_dir = Path('../results')
results_dir.mkdir(exist_ok=True)

# 1) Luu ca 2 model (chua chon "best" o day - de quyet dinh sau khi xem so lieu that)
joblib.dump(xgb_plain, results_dir / 'xgb_plain_model.pkl')
joblib.dump(xgb_smote, results_dir / 'xgb_smote_model.pkl')
print('Da luu 2 model vao ../results/')

# 2) Luu bang so sanh XGBoost (plain vs SMOTE) ra CSV - dung truc tiep cho report
xgb_comparison.to_csv(results_dir / 'xgb_vs_xgb_smote_comparison.csv', index=False)
print('Da luu bang so sanh XGBoost vao ../results/xgb_vs_xgb_smote_comparison.csv')

# 3) Luu ca X_train_smote/y_train_smote neu Tuan 4 can dung lai (vd de train them variant khac)
joblib.dump(
    {'X_train_smote': X_train_smote, 'y_train_smote': y_train_smote},
    results_dir / 'train_smote_data.pkl'
)
print('Da luu du lieu SMOTE vao ../results/train_smote_data.pkl')

# 4) In lai tom tat de bo vao report muc "Extension 1"
print('\n=== TOM TAT DE VIET REPORT ===')
print(f"XGBoost (plain)   - F1: {f1_plain:.4f}, Recall: {recall_plain:.4f}")
print(f"XGBoost + SMOTE   - F1: {f1_smote:.4f}, Recall: {recall_smote:.4f}")
print(f"SMOTE cai thien Recall: {'CO' if recall_smote > recall_plain else 'KHONG'} "
      f"({recall_smote - recall_plain:+.4f})")